In [1]:
# Social Media Sentiment Analysis (GitHub-ready)

import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

print("Loading dataset from Google Drive...")

drive_link = "https://drive.google.com/file/d/16bUuoTpAbqnorkZXbSUtqfH5OdtAa9a5/view?usp=sharing"

file_id = re.search(r'/d/([a-zA-Z0-9_-]+)', drive_link).group(1)
url = f"https://drive.google.com/uc?id={file_id}"

data = pd.read_csv(url, header=None)

print(f"Dataset loaded: {len(data)} rows")

Loading dataset from Google Drive...
Dataset loaded: 74682 rows


In [2]:
# Select label and text columns
data = data[[2,3]]
data = data.rename(columns={2:'label',3:'text'})

In [3]:
# Clean labels
data['label'] = data['label'].astype(str).str.strip().str.capitalize()

valid_labels = ["Positive","Negative","Neutral"]
data = data[data['label'].isin(valid_labels)]

print("\nLabel distribution:")
print(data['label'].value_counts())


Label distribution:
label
Negative    22542
Positive    20832
Neutral     18318
Name: count, dtype: int64


In [4]:
# Balance dataset
min_count = int(data['label'].value_counts().min())

balanced_data = pd.concat([
    data[data['label']=="Positive"].sample(min_count, random_state=42),
    data[data['label']=="Negative"].sample(min_count, random_state=42),
    data[data['label']=="Neutral"].sample(min_count, random_state=42)
])

print("\nBalanced dataset size:", len(balanced_data))


Balanced dataset size: 54954


In [5]:
#  FIX: remove empty text rows
balanced_data = balanced_data.dropna(subset=["text"])

In [6]:
# Convert to string
balanced_data["text"] = balanced_data["text"].astype(str)

In [7]:
# Text Vectorization
vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)

X = vectorizer.fit_transform(balanced_data["text"])
y = balanced_data["label"]

In [8]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\nTraining model...")

model = MultinomialNB()
model.fit(X_train, y_train)


Training model...


MultinomialNB()

In [9]:
# Evaluation
predictions = model.predict(X_test)

accuracy = round(accuracy_score(y_test, predictions) * 100, 2)

print("\nModel Accuracy:", accuracy, "%")


Model Accuracy: 72.28 %


In [10]:
# Custom Predictions
print("\nCustom Predictions:")

test_posts = [
    "I love this game!",
    "Not sure about this update.",
    "This is terrible!"
]

test_vector = vectorizer.transform(test_posts)

for text, pred in zip(test_posts, model.predict(test_vector)):
    print(text, "=>", pred)


Custom Predictions:
I love this game! => Positive
Not sure about this update. => Negative
This is terrible! => Negative
